In [0]:
%run ./1_config

nyc_taxi


catalog : nyc_taxi
schemas : landing / bronze / silver / gold
landing volume : /Volumes/nyc_taxi/landing/files


In [0]:
import pyspark.sql.functions as F

## Bronze: Trips data (Parquet) 

In [0]:
df_bronze_trips = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{VOLUME_ROOT}/_SCHEMA/trips")
    .load(RAW_TRIPS)
    .withColumn("_ingest_time", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_process_ingestion", F.lit("Auto_Loader"))
)

In [0]:
q_trips = (
    df_bronze_trips.writeStream
    .format("delta")
    .option("checkpointLocation", f"{VOLUME_ROOT}/checkpoints/bronze_trips/")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(T_BRONZE_TRIPS)
)
q_trips.awaitTermination()

In [0]:
print("Bronze trips rows:", spark.table(T_BRONZE_TRIPS).count())

Bronze trips rows: 2964624


In [0]:
display(spark.table(T_BRONZE_TRIPS).limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,_rescued_data,_ingest_time,_source_file,_process_ingestion
2,2024-01-01T00:57:55.000,2024-01-01T01:17:43.000,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:03:00.000,2024-01-01T00:09:36.000,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:17:06.000,2024-01-01T00:35:01.000,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:36:38.000,2024-01-01T00:44:56.000,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:46:51.000,2024-01-01T00:52:57.000,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:54:08.000,2024-01-01T01:26:31.000,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
2,2024-01-01T00:49:44.000,2024-01-01T01:15:47.000,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
1,2024-01-01T00:30:40.000,2024-01-01T00:58:40.000,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
2,2024-01-01T00:26:01.000,2024-01-01T00:54:12.000,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader
2,2024-01-01T00:28:08.000,2024-01-01T00:29:16.000,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,null,2026-10-15T08:45:05.768Z,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,Auto_Loader


## Bronze: Zones (CSV)

In [0]:
df_bronze_zones = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{VOLUME_ROOT}/_SCHEMA/zones")
    .load(RAW_ZONES)
    .withColumn("_ingest_time", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_process_ingestion", F.lit("Auto_Loader"))
)

In [0]:
q_zones = (
    df_bronze_zones.writeStream
    .format("delta")
    .option("checkpointLocation", f"{VOLUME_ROOT}/checkpoints/bronze_zones/")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(T_BRONZE_ZONES)
)
q_zones.awaitTermination()
     

In [0]:
print("Bronze zones rows:", spark.table(T_BRONZE_ZONES).count())

Bronze zones rows: 265


In [0]:
display(spark.table(T_BRONZE_ZONES).limit(10))

LocationID,Borough,Zone,service_zone,_rescued_data,_ingest_time,_source_file,_process_ingestion
1,EWR,Newark Airport,EWR,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
2,Queens,Jamaica Bay,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
3,Bronx,Allerton/Pelham Gardens,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
4,Manhattan,Alphabet City,Yellow Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
5,Staten Island,Arden Heights,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
7,Queens,Astoria,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
8,Queens,Astoria Park,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
9,Queens,Auburndale,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
10,Queens,Baisley Park,Boro Zone,null,2026-10-15T08:45:24.437Z,/Volumes/nyc_taxi/landing/files/raw_zones/taxi_zone_lookup.csv,Auto_Loader
